
# Guía 2 - Selección de atributos, revisión de sesgo y balanceo

Este notebook plantilla acompaña la Guía 2. El objetivo es preparar un dataset más confiable antes del modelado, aplicando:

- identificación de variable objetivo;
- revisión de desbalance;
- revisión de representatividad;
- selección de atributos;
- balanceo solo del conjunto de entrenamiento;
- exportación de datasets finales.

**Recordatorio de calidad:** Segmento es ordinal: Básico < Medio < Premium. Satisfacción también es ordinal: Baja < Media < Alta.


# IMPORTAR LIBRERIAS 

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif 
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import balanced_accuracy_score, f1_score 
from sklearn.utils import resample




# CARGAR DATASET

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/jaidisplata/PRODEMIA/refs/heads/main/2-Carpeta_Estudiantes_Guia2_Seleccion_Atributos_Sesgo_Balanceo/02_Datos/dataset_clientes_guia2_base.csv"

df = pd.read_csv(url)   

df.head()  


,ID_Cliente,Edad,IngresoMensual,CantidadCompras,ComprasUltimos12M,AntiguedadMeses,QuejasUltimos6M,DiasDesdeUltimaCompra,VisitasWebUltimoMes,TiempoPromedioSesionMin,CuponesUsados,Ciudad,CanalPreferido,ZonaResidencia,Segmento,Satisfaccion,CodigoCampania,Abandono
0,C2000,18,1842000,2,1,10,0,36,0,7.3,2,Bogotá,Tienda física,Rural,Básico,Alta,B,0
1,C2001,33,2326000,2,4,7,0,38,5,9.2,1,Barranquilla,Web,Urbana,Básico,Media,A,0
2,C2002,48,5048000,8,9,16,0,6,8,8.0,0,Medellín,Referido,Urbana,Medio,Alta,A,0
3,C2003,53,3641000,4,5,7,1,29,4,5.7,1,Cali,Tienda física,Urbana,Medio,Baja,D,0
4,C2004,48,2144000,4,4,34,0,42,2,8.3,4,Cali,Redes sociales,Rural,Básico,Alta,B,0


# DIAGNÓSTICO INICIAL

In [3]:
# 2. Diagnóstico inicial
print(df.shape)
df.info()


(360, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID_Cliente               360 non-null    object 
 1   Edad                     360 non-null    int64  
 2   IngresoMensual           360 non-null    int64  
 3   CantidadCompras          360 non-null    int64  
 4   ComprasUltimos12M        360 non-null    int64  
 5   AntiguedadMeses          360 non-null    int64  
 6   QuejasUltimos6M          360 non-null    int64  
 7   DiasDesdeUltimaCompra    360 non-null    int64  
 8   VisitasWebUltimoMes      360 non-null    int64  
 9   TiempoPromedioSesionMin  360 non-null    float64
 10  CuponesUsados            360 non-null    int64  
 11  Ciudad                   360 non-null    object 
 12  CanalPreferido           360 non-null    object 
 13  ZonaResidencia           360 non-null    object 
 14  Segmento        

# DEFINIR VARIABLE OBJETIVO Y DESBALANCE


In [4]:
# 3. Variable objetivo y desbalance
objetivo = 'Abandono'
print(df[objetivo].value_counts())
print(df[objetivo].value_counts(normalize=True).round(4))



Abandono
0    268
1     92
Name: count, dtype: int64
Abandono
0    0.7444
1    0.2556
Name: proportion, dtype: float64


# CLASIFICACIÓN TÉCNICA DE VARIABLES**

In [5]:
# 4. Clasificación técnica de variables
id_col = 'ID_Cliente'
numeric_cols = ['Edad', 'IngresoMensual', 'CantidadCompras', 'ComprasUltimos12M', 'AntiguedadMeses','QuejasUltimos6M','DiasDesdeUltimaCompra','VisitasWebUltimoMes','TiempoPromedioSesionMin','CuponesUsados']

nominal_cols = ['Ciudad','CanalPreferido','ZonaResidencia','CodigoCampania']
ordinal_cols = ['Segmento','Satisfaccion']
ordinal_categories = [['Básico','Medio','Premium'],['Baja','Media','Alta']]

print('Numéricas:', numeric_cols)
print('Nominales:', nominal_cols)
print('Ordinales:', ordinal_cols)   

Numéricas: ['Edad', 'IngresoMensual', 'CantidadCompras', 'ComprasUltimos12M', 'AntiguedadMeses', 'QuejasUltimos6M', 'DiasDesdeUltimaCompra', 'VisitasWebUltimoMes', 'TiempoPromedioSesionMin', 'CuponesUsados']
Nominales: ['Ciudad', 'CanalPreferido', 'ZonaResidencia', 'CodigoCampania']
Ordinales: ['Segmento', 'Satisfaccion']


# REVISIÓN DE REPRESENTATIVIDAD

In [6]:
# 5. Revisión de representatividad por zona
pd.crosstab(df['ZonaResidencia'], df[objetivo], normalize='index').round(3)


Abandono,0,1
ZonaResidencia,,
Periurbana,0.689,0.311
Rural,0.623,0.377
Urbana,0.790,0.210


# SEPARACIÓN, ENTRENAMIENTO Y PRUEBA ESTRATIFICADA

In [7]:
# 6. Separación entrenamiento/prueba estratificada
x = df.drop(columns=[objetivo, id_col])
y = df[objetivo]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

print('Distribución entrenamiento:')
print(y_train.value_counts(normalize=True).round(3))
print('Distribución prueba:')
print(y_test.value_counts(normalize=True).round(3))


Distribución entrenamiento:
Abandono
0    0.744
1    0.256
Name: proportion, dtype: float64
Distribución prueba:
Abandono
0    0.744
1    0.256
Name: proportion, dtype: float64


# PREPROCESAMIENTO SIN FUGA DE INFORMACIÓN

In [8]:
# 7. Preprocesamiento sin fuga de información
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False),nominal_cols),
        ('ord', OrdinalEncoder(categories=ordinal_categories), ordinal_cols),], remainder='drop', verbose_feature_names_out=False)

x_train_proc = preprocessor.fit_transform(x_train)
x_test_proc = preprocessor.transform(x_test)

feature_names = preprocessor.get_feature_names_out()
x_train_df = pd.DataFrame(x_train_proc, columns=feature_names, index=x_train.index)
x_test_df = pd.DataFrame(x_test_proc, columns=feature_names, index=x_test.index)

x_train_df.head()




,Edad,IngresoMensual,CantidadCompras,ComprasUltimos12M,AntiguedadMeses,QuejasUltimos6M,DiasDesdeUltimaCompra,VisitasWebUltimoMes,TiempoPromedioSesionMin,CuponesUsados,...,ZonaResidencia_Rural,ZonaResidencia_Urbana,CodigoCampania_A,CodigoCampania_B,CodigoCampania_C,CodigoCampania_D,CodigoCampania_E,CodigoCampania_F,Segmento,Satisfaccion
12,-1.584614,0.181255,0.175338,-0.338109,0.438723,0.1828,-0.973211,0.646622,-0.104759,-0.077344,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
14,-0.353698,2.234681,-0.655211,-0.074267,1.810525,3.8388,1.030584,-0.676014,-0.780892,-0.691545,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
331,-0.866580,0.211077,-0.932061,-0.074267,-0.658719,-0.7312,-0.857607,1.969258,-0.968707,-0.077344,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0
267,-0.353698,1.547569,0.452188,-0.074267,-0.384359,1.0968,-1.320021,-0.014696,1.397759,1.151059,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
245,0.569488,0.209251,0.729038,0.981101,2.084885,0.1828,4.074811,-1.006673,1.698263,-0.691545,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0


# REVISIÓN DE BAJA VARIABILIDAD

In [9]:
# 8. Revisión de baja variabilidad
var_selector = VarianceThreshold(threshold=0.005)
var_selector.fit(x_train_df)
var_support = pd.Series(var_selector.get_support(), index=x_train_df.columns)
var_support = pd.Series(var_selector.get_support(), index=x_train_df.columns)
var_support.value_counts()



                                 

True    30
Name: count, dtype: int64

# REVISIÓN DE CORRELACIÓN ENTRE VARIABLES NUMÉRICAS

In [10]:
# 9. Revisión de correlación entre variables numéricas
corr = x_train[numeric_cols].corr().abs()
pares_redundantes = []
for i, c1 in enumerate(corr.columns):
    for c2 in corr.columns[i+1:]:
        if corr.loc[c1, c2] >= 0.85:
            pares_redundantes.append((c1, c2, corr.loc[c1, c2]))
            
pares_redundantes


[('CantidadCompras', 'ComprasUltimos12M', 0.9320308628838077)]

# MÉTODO F-SCORE

In [11]:
# 10. Método de filtro: F-score
selector = SelectKBest(score_func=f_classif, k=min(12, x_train_df.shape[1]))
selector.fit(x_train_df, y_train)
ranking_f = pd.DataFrame({'Atributo': x_train_df.columns, 'Puntaje_F': selector.scores_,'p_value': selector.pvalues_}).sort_values('Puntaje_F',ascending=False)

ranking_f.head(12)



,Atributo,Puntaje_F,p_value
29,Satisfaccion,23.118657,0.000003
5,QuejasUltimos6M,19.805203,0.000013
28,Segmento,13.059031,0.000360
9,CuponesUsados,5.143341,0.024132
21,ZonaResidencia_Urbana,4.912848,0.027497
0,Edad,3.263206,0.071972
6,DiasDesdeUltimaCompra,3.168740,0.076193
20,ZonaResidencia_Rural,2.485780,0.116059
3,ComprasUltimos12M,2.328095,0.128236
2,CantidadCompras,2.190334,0.140054


# MÉTODO INTEGRADO CON RANDOM FOREST

In [12]:
# 11. Método integrado: importancia con Random Forest
rf = RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced', max_depth=5)
rf.fit(x_train_df, y_train)
ranking_rf = pd.DataFrame({'Atributo': x_train_df.columns, 
    'Importancia_RF': rf.feature_importances_}).sort_values('Importancia_RF', ascending=False)

ranking_rf.head(12)



,Atributo,Importancia_RF
29,Satisfaccion,0.101344
0,Edad,0.087642
1,IngresoMensual,0.086302
4,AntiguedadMeses,0.078782
5,QuejasUltimos6M,0.073620
8,TiempoPromedioSesionMin,0.070409
6,DiasDesdeUltimaCompra,0.062680
9,CuponesUsados,0.057665
3,ComprasUltimos12M,0.057360
2,CantidadCompras,0.053193


# SELECCIÓN FINAL DE ATRIBUTOS

In [13]:
# 12. Selección final de atributos
seleccionados = list(dict.fromkeys(list(ranking_f.head(10)['Atributo']) + list(ranking_rf.head(10)['Atributo']))) 
seleccionados = [s for s in seleccionados if var_support.get(s, True)][:14]
# Control de redundancia: si ambas están, se conserva CantidadCompras 
if 'CantidadCompras' in seleccionados and 'ComprasUltimos12M' in seleccionados: 
    seleccionados.remove('ComprasUltimos12M')

seleccionados


['Satisfaccion',
 'QuejasUltimos6M',
 'Segmento',
 'CuponesUsados',
 'ZonaResidencia_Urbana',
 'Edad',
 'DiasDesdeUltimaCompra',
 'ZonaResidencia_Rural',
 'CantidadCompras',
 'IngresoMensual',
 'AntiguedadMeses',
 'TiempoPromedioSesionMin']

# CONSTRUCCIÓN DE DATASET SELECCIONADO

In [14]:
# 13. Construir dataset seleccionado
train_selected = x_train_df[seleccionados].copy()
train_selected[objetivo] = y_train.values 

test_selected = x_test_df[seleccionados].copy() 
test_selected[objetivo] = y_test.values 

train_selected.head()



,Satisfaccion,QuejasUltimos6M,Segmento,CuponesUsados,ZonaResidencia_Urbana,Edad,DiasDesdeUltimaCompra,ZonaResidencia_Rural,CantidadCompras,IngresoMensual,AntiguedadMeses,TiempoPromedioSesionMin,Abandono
12,1.0,0.1828,0.0,-0.077344,1.0,-1.584614,-0.973211,0.0,0.175338,0.181255,0.438723,-0.104759,1
14,0.0,3.8388,1.0,-0.691545,0.0,-0.353698,1.030584,0.0,-0.655211,2.234681,1.810525,-0.780892,1
331,2.0,-0.7312,0.0,-0.077344,1.0,-0.866580,-0.857607,0.0,-0.932061,0.211077,-0.658719,-0.968707,1
267,0.0,1.0968,0.0,1.151059,1.0,-0.353698,-1.320021,0.0,0.452188,1.547569,-0.384359,1.397759,1
245,1.0,0.1828,1.0,-0.691545,0.0,0.569488,4.074811,0.0,0.729038,0.209251,2.084885,1.698263,0


# BALANCEO DE CONJUNTO DE ENTRENAMIENTO

In [15]:
# 14. Balanceo solo del conjunto de entrenamiento
conteo_antes = train_selected[objetivo].value_counts().sort_index()
print(conteo_antes)

mayoritaria = train_selected[train_selected[objetivo] ==conteo_antes.idxmax()]

minoritaria = train_selected[train_selected[objetivo] ==conteo_antes.idxmax()]

minoritaria_up = resample(
    minoritaria, replace=True,
    n_samples=len(mayoritaria),
    random_state=42)

train_balanced = pd.concat([mayoritaria, minoritaria_up]).sample(frac=1, random_state=42).reset_index(drop=True)

print('Después de balancear:')
print(train_balanced[objetivo].value_counts().sort_index())



Abandono
0    201
1     69
Name: count, dtype: int64
Después de balancear:
Abandono
0    402
Name: count, dtype: int64


# VERIFICACIÓN CON MODELO BÁSICO

In [16]:
# 15. Verificación rápida con modelo básico 
modelo_sin_balanceo = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=5)
modelo_sin_balanceo.fit(train_selected.drop(columns=[objetivo]), train_selected[objetivo])
pred_sin = modelo_sin_balanceo.predict(test_selected.drop(columns=[objetivo]))
modelo_balanceado = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=5)
modelo_balanceado.fit(train_balanced.drop(columns=[objetivo]), train_balanced[objetivo]) 
pred_bal = modelo_balanceado.predict(test_selected.drop(columns=[objetivo]))

metricas = pd.DataFrame([
    
     { 
        'Escenario':'Sin balanceo', 
        'Balanced Accuracy': balanced_accuracy_score(test_selected[objetivo], pred_sin), 
        'F1 clase 1': f1_score(test_selected[objetivo], pred_sin, pos_label=1, zero_division=0) 
    }, 
    {  
            'Escenario':'Con balanceo en entrenamiento',
            'Balanced Accuracy': balanced_accuracy_score(test_selected[objetivo], pred_bal),
            'F1 clase 1': f1_score(test_selected[objetivo], pred_bal, pos_label=1, zero_division=0)
 
    }  
])  
metricas

         



,Escenario,Balanced Accuracy,F1 clase 1
0,Sin balanceo,0.601233,0.344828
1,Con balanceo en entrenamiento,0.500000,0.000000


# ENTREGABLES PARA EXPORTAR

In [17]:
# 16. Exportar entregables
ranking_f.to_csv('ranking_atributos_fscore_guia2.csv', index=False, encoding= 'utf-8-sig')
ranking_rf.to_csv('ranking_atributos_random_forest_guia2.csv', index=False, encoding='utf-8-sig')
train_selected.to_csv('dataset_train_seleccionado_balanceado_guia2.csv', index=False, encoding='utf-8-sig')
test_selected.to_csv('dataset_testseleccionado_guia2.csv', index=False, encoding= 'utf-8-sig')
metricas.to_csv('metricas_verificacion_balanceo_guia2.csv', index=False, encoding='utf-8-sig')

print('Entregables exportados correctamente.')




Entregables exportados correctamente.


In [1]:
import os
print(os.getcwd())

c:\Users\HP\Desktop\EntregablesMayoH


# VERIFICACIÓN DE ENTREGABLES 

In [19]:
train_selected.to_csv('train_selected.csv', index=False)
test_selected.to_csv('test_selected.csv', index=False)
metricas.to_csv('metricas.csv', index=False)

In [20]:
pd.read_csv('train_selected.csv').head()

,Satisfaccion,QuejasUltimos6M,Segmento,CuponesUsados,ZonaResidencia_Urbana,Edad,DiasDesdeUltimaCompra,ZonaResidencia_Rural,CantidadCompras,IngresoMensual,AntiguedadMeses,TiempoPromedioSesionMin,Abandono
0,1.0,0.1828,0.0,-0.077344,1.0,-1.584614,-0.973211,0.0,0.175338,0.181255,0.438723,-0.104759,1
1,0.0,3.8388,1.0,-0.691545,0.0,-0.353698,1.030584,0.0,-0.655211,2.234681,1.810525,-0.780892,1
2,2.0,-0.7312,0.0,-0.077344,1.0,-0.866580,-0.857607,0.0,-0.932061,0.211077,-0.658719,-0.968707,1
3,0.0,1.0968,0.0,1.151059,1.0,-0.353698,-1.320021,0.0,0.452188,1.547569,-0.384359,1.397759,1
4,1.0,0.1828,1.0,-0.691545,0.0,0.569488,4.074811,0.0,0.729038,0.209251,2.084885,1.698263,0


In [21]:
pd.read_csv('test_selected.csv').head()

,Satisfaccion,QuejasUltimos6M,Segmento,CuponesUsados,ZonaResidencia_Urbana,Edad,DiasDesdeUltimaCompra,ZonaResidencia_Rural,CantidadCompras,IngresoMensual,AntiguedadMeses,TiempoPromedioSesionMin,Abandono
0,2.0,-0.7312,1.0,-0.691545,1.0,0.159183,-0.048382,0.0,-0.378361,1.619993,1.124624,-1.569714,0
1,2.0,-0.7312,0.0,1.151059,1.0,-0.045969,-1.358556,0.0,-0.655211,0.506249,-0.247178,-0.367700,0
2,1.0,2.9248,0.0,-0.691545,0.0,0.979794,0.529635,0.0,-1.208910,-0.785814,0.781673,-0.480388,1
3,2.0,0.1828,0.0,2.379461,0.0,1.390099,0.452566,1.0,-0.655211,-0.182080,-0.247178,0.984567,0
4,2.0,-0.7312,2.0,-1.305747,0.0,0.979794,-0.510796,0.0,0.452188,3.570565,-0.590129,1.097256,0


In [22]:
pd.read_csv('metricas.csv')

,Escenario,Balanced Accuracy,F1 clase 1
0,Sin balanceo,0.601233,0.344828
1,Con balanceo en entrenamiento,0.500000,0.000000
